# 07 — Development comparison and fail-closed selection

This is the only model-selection notebook allowed to open development
truth. Threshold values come from one late-calibration half. The other half
filters out operating points that cannot satisfy the workload budget without
labels. Development then selects among those admissible operating points;
this is ordinary validation, while holdout remains the untouched final test.
A threshold is selectable only when at least five calibration block
maxima are expected beyond its quantile. Diagnostic topology and multivariate
ablations remain visible but cannot be selected unless the policy explicitly
marks them eligible.

Holdout remains physically sealed. Workload rates use the time where at
least one selected channel is actually scoreable, while calendar exposure
is reported beside it. Three estimands are kept separate: detection during
the active fault, warning before impact, and prompt detection within 48 hours.
All recall gates use 95% Wilson lower bounds. Detection, early-warning and
localisation qualifications are reported separately.

## 1. Setup and evaluation boundary


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import time

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

from telco_anomaly.pipeline.materialize import partition_exposure
from telco_anomaly.scoring import duration_to_observations
from telco_anomaly.scoring.alerts import alert_grid_from_score_file
from telco_anomaly.evaluation import (
    evaluate_cases, form_cases, poisson_rate_interval,
    scoreable_exposures, wilson_interval,
)
from telco_anomaly.selection import (
    qualify_early_warning, qualify_localisation, select_development_candidate,
)
from telco_anomaly.io import (
    file_sha256,
    immutable_output_directory,
    read_json,
    require_same,
    resolve_data_root,
    source_tree_sha256,
    write_json,
)

DATA_ROOT = resolve_data_root()
CORE_RUN_ID = os.getenv(
    "TELCO_CORE_RUN_ID", os.getenv("PON_CORE_RUN_ID", "synthetic_pon_core_v2")
)
MODEL_RUN_ID = os.getenv(
    "TELCO_MODEL_RUN_ID", os.getenv("PON_MODEL_RUN_ID", "synthetic_pon_models_v11")
)
TRUTH_RUN_ID = os.getenv("PON_TRUTH_RUN_ID", "synthetic_pon_truth_v3")
SELECTION_RUN_ID = os.getenv("PON_SELECTION_RUN_ID", "synthetic_pon_selection_v13")

RUN_ROOT = DATA_ROOT / "core" / "synthetic_pon" / CORE_RUN_ID
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
MODEL_ROOT = DATA_ROOT / "models" / "synthetic_pon" / MODEL_RUN_ID
TRUTH_ROOT = DATA_ROOT / "evaluation" / "synthetic_pon" / TRUTH_RUN_ID
DEV_TRUTH = TRUTH_ROOT / "development"
HOLDOUT_TRUTH = TRUTH_ROOT / "holdout_locked"
OUTPUT_ROOT = DATA_ROOT / "selection" / "synthetic_pon" / SELECTION_RUN_ID

core_manifest = read_json(CORE_ROOT / "manifest.json")
model_manifest = read_json(MODEL_ROOT / "model_manifest.json")
resolved_policy = read_json(MODEL_ROOT / "resolved_policy.json")
truth_manifest = read_json(TRUTH_ROOT / "truth_manifest.json")
require_same(model_manifest, core_fingerprint=core_manifest["fingerprint"])
require_same(
    truth_manifest, model_core_fingerprint=core_manifest["fingerprint"]
)
if source_tree_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly",
    patterns=("detectors.py", "scoring/*.py", "pipeline/*.py"),
) != model_manifest["detector_package_sha256"]:
    raise ValueError("Detector code changed after the model was fitted")
EVALUATION_MODULE_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "evaluation.py"
)
SELECTION_MODULE_SHA256 = file_sha256(
    PROJECT_ROOT / "src" / "telco_anomaly" / "selection.py"
)
if file_sha256(MODEL_ROOT / "resolved_policy.json") != model_manifest["resolved_policy_sha256"]:
    raise ValueError("The frozen model policy no longer matches its manifest")
POLICY = resolved_policy["alert_policy"]
score_path = MODEL_ROOT / model_manifest["score_files"]["development"]
threshold_score_path = (
    MODEL_ROOT / model_manifest["score_files"]["calibration_threshold"]
)
verification_score_path = (
    MODEL_ROOT / model_manifest["score_files"]["calibration_verification"]
)
thresholds = pd.read_parquet(MODEL_ROOT / "calibration_thresholds.parquet")
topology_path = CORE_ROOT / "topology_memberships.parquet"
topology = pd.read_parquet(topology_path) if topology_path.exists() else pd.DataFrame()

if not DEV_TRUTH.is_dir():
    raise FileNotFoundError("Run Notebook 03 to create development truth")
for name in ("fault_events.parquet", "fault_entity_intervals.parquet"):
    relative = f"development/{name}"
    if file_sha256(DEV_TRUTH / name) != truth_manifest["file_sha256"].get(relative):
        raise ValueError(f"Development truth file changed: {relative}")
events = pd.read_parquet(DEV_TRUTH / "fault_events.parquet")
intervals = pd.read_parquet(DEV_TRUTH / "fault_entity_intervals.parquet")

display(pd.Series({
    "scores": str(score_path),
    "truth_opened": "development only",
    "holdout_opened": False,
    "development_faults": events["fault_id"].nunique(),
}, name="value").to_frame())


## 2. Candidate portfolios and operational persistence

The existing CUSUM is the slow-change path: it accumulates evidence and
requires one observed threshold crossing. The soft-confirmed Isolation
Forest is a fast-path challenger with one-observation confirmation. Both
must pass the same independent false-case workload gate; neither is
selected because it looks better on individual faults.


In [ ]:
CADENCE_SECONDS = float(model_manifest["cadence_seconds"])
aliases = {
    "rapid_self": "rapid_residual",
    "persistent_drift": "drift_cusum",
}

declared = {
    **POLICY["portfolios"],
    **POLICY.get("diagnostic_portfolios", {}),
}
available = set(model_manifest["channels"])
portfolios = {}
for name, declared_channels in declared.items():
    channels = [aliases.get(channel, channel) for channel in declared_channels]
    if set(channels) <= available:
        portfolios[name] = channels

eligible_portfolios = set(POLICY["selection"]["eligible_portfolios"])
selection_eligible = {
    name: name in eligible_portfolios for name in portfolios
}

channel_policy = {
    aliases.get(name, name): values
    for name, values in POLICY["channels"].items()
}
missing_persistence = available - set(channel_policy)
if missing_persistence:
    raise ValueError(
        f"Missing persistence policy for {sorted(missing_persistence)}"
    )
persistence = {
    channel: int(channel_policy[channel]["persistence_observations"])
    if "persistence_observations" in channel_policy[channel]
    else duration_to_observations(
        channel_policy[channel]["persistence_seconds"], CADENCE_SECONDS
    )
    for channel in available
}
RECOVERY_OBSERVATIONS = duration_to_observations(
    POLICY["recovery"]["duration_seconds"], CADENCE_SECONDS
)
RECOVERY_THRESHOLD_FRACTION = float(
    POLICY["recovery"]["threshold_fraction"]
)

display(pd.DataFrame([
    {
        "portfolio": name,
        "channels": ", ".join(channels),
        "selection_eligible": selection_eligible[name],
    }
    for name, channels in portfolios.items()
]))

## 3. Build alerts once per channel

Thresholds are calibration-frozen. Development labels are used only after the
score-to-alert transformation, and only for operating points that passed the
independent calibration workload check.


In [ ]:
verification_grid = alert_grid_from_score_file(
    verification_score_path,
    thresholds,
    persistence=persistence,
    recovery_consecutive=RECOVERY_OBSERVATIONS,
    cadence_seconds=CADENCE_SECONDS,
    recovery_threshold_fraction=RECOVERY_THRESHOLD_FRACTION,
)

calendar_exposure = partition_exposure(
    score_path, "entity_day", CADENCE_SECONDS
)
verification_calendar_exposure = partition_exposure(
    verification_score_path, "entity_day", CADENCE_SECONDS
)


def score_missingness(path, channels):
    source = str(path).replace("'", "''")
    channels = sorted(channels)
    aliases = {channel: f"missing_{index}" for index, channel in enumerate(channels)}
    entity_rates = []
    summaries = []
    for channel, alias in aliases.items():
        quoted = '"' + channel.replace('"', '""') + '"'
        entity_rates.append(
            f"avg(CASE WHEN {quoted} IS NULL OR "
            f"NOT isfinite(CAST({quoted} AS DOUBLE)) THEN 1.0 ELSE 0.0 END) "
            f"AS {alias}"
        )
        summaries.extend([f"avg({alias})", f"quantile_cont({alias}, 0.95)"])
    with duckdb.connect() as connection:
        values = connection.execute(f"""
            WITH entity_missingness AS (
                SELECT CAST(entity_id AS VARCHAR) AS entity_id,
                       {', '.join(entity_rates)}
                FROM read_parquet('{source}')
                GROUP BY entity_id
            )
            SELECT {', '.join(summaries)} FROM entity_missingness
        """).fetchone()
    result = {}
    for index, channel in enumerate(channels):
        result[channel] = {
            "global": float(values[2 * index] or 0.0),
            "entity_p95": float(values[2 * index + 1] or 0.0),
        }
    return result


def threshold_exceedance_audit(path, threshold_table):
    source = str(path).replace("'", "''")
    rows = []
    with duckdb.connect() as connection:
        for channel, group in threshold_table.groupby("model_id", sort=False):
            group = group.sort_values("threshold_quantile")
            quoted = '"' + str(channel).replace('"', '""') + '"'
            expressions = [f"count({quoted})"]
            parameters = []
            for row in group.itertuples(index=False):
                expressions.append(
                    f"sum(CASE WHEN {quoted} >= ? THEN 1 ELSE 0 END)"
                )
                parameters.append(float(row.threshold))
            values = connection.execute(
                f"SELECT {', '.join(expressions)} FROM read_parquet('{source}')",
                parameters,
            ).fetchone()
            valid_scores = int(values[0])
            previous_threshold = None
            for row, exceedances in zip(group.itertuples(index=False), values[1:]):
                rows.append({
                    "model_id": channel,
                    "threshold_quantile": float(row.threshold_quantile),
                    "threshold": float(row.threshold),
                    "blocks_used": int(row.blocks_used),
                    "unique_block_maxima": int(row.unique_block_maxima),
                    "expected_tail_blocks": float(row.expected_tail_blocks),
                    "threshold_tied_to_previous": (
                        previous_threshold is not None
                        and float(row.threshold) == previous_threshold
                    ),
                    "valid_scores": valid_scores,
                    "raw_exceedances": int(exceedances or 0),
                    "raw_exceedance_fraction": (
                        float(exceedances or 0) / valid_scores
                        if valid_scores else float("nan")
                    ),
                })
                previous_threshold = float(row.threshold)
    audit = pd.DataFrame(rows)
    for channel, group in audit.groupby("model_id"):
        ordered = group.sort_values("threshold_quantile")
        if ordered["threshold"].diff().dropna().lt(-1e-12).any():
            raise AssertionError(f"Thresholds decrease for {channel}")
        if ordered["raw_exceedances"].diff().dropna().gt(0).any():
            raise AssertionError(f"Exceedances increase for {channel}")
    return audit


score_coverage = score_missingness(score_path, available)
threshold_audit = threshold_exceedance_audit(
    threshold_score_path, thresholds
)
display(threshold_audit)
print(f"Threshold-estimation source: {threshold_score_path.name}")
print(
    f"Independent workload calendar exposure: "
    f"{verification_calendar_exposure:,.1f} entity-days"
)
print(f"Development calendar exposure: {calendar_exposure:,.1f} entity-days")

## 4. Compare at a matched incident workload


In [ ]:
def metric_row(result, name):
    row = result["metrics"].loc[result["metrics"]["metric"].eq(name)]
    if row.empty:
        raise KeyError(f"Evaluation did not return {name!r}")
    return row.iloc[0]


def threshold_map(channels, quantile):
    return {
        channel: float(thresholds.loc[
            thresholds["model_id"].eq(channel)
            & thresholds["threshold_quantile"].eq(quantile),
            "threshold",
        ].iloc[0])
        for channel in channels
    }


def cases_from_grid(grid, channels, quantile):
    alerts = pd.concat(
        [grid[(channel, float(quantile))] for channel in channels],
        ignore_index=True,
    )
    if len(alerts):
        alerts = alerts.sort_values("alert_start").reset_index(drop=True)
        alerts["alert_id"] = [
            f"A-{number:09d}" for number in range(1, len(alerts) + 1)
        ]
    cases, members = form_cases(
        alerts,
        topology,
        gap_seconds=POLICY["incidents"]["quiet_period_seconds"],
        thresholds=threshold_map(channels, quantile),
        shared_scope_models=("group_common_mode",),
    )
    return alerts, cases, members


quantiles = sorted(thresholds["threshold_quantile"].unique())
budget = float(POLICY["workload"]["false_incidents_per_entity_day"])
budget_gate = budget * float(POLICY["workload"]["safety_factor"])
confidence = float(POLICY["workload"]["confidence_level"])
minimum_tail_blocks = float(
    POLICY["thresholds"]["minimum_expected_tail_blocks"]
)
verification_workload = {}
verification_rows = []
admissible_quantiles = {}
label_free_quantile = {}
verification_scoreable_exposure = scoreable_exposures(
    verification_score_path, portfolios, "entity_day", CADENCE_SECONDS
)
development_scoreable_exposure = scoreable_exposures(
    score_path, portfolios, "entity_day", CADENCE_SECONDS
)
exposure_audit = pd.DataFrame([
    {
        "partition": partition,
        "portfolio": portfolio,
        "calendar_entity_days": calendar_days,
        "scoreable_entity_days": scoreable[portfolio],
        "unscoreable_entity_days": calendar_days - scoreable[portfolio],
        "score_availability": (
            scoreable[portfolio] / calendar_days if calendar_days else float("nan")
        ),
    }
    for partition, calendar_days, scoreable in (
        ("calibration_verification", verification_calendar_exposure, verification_scoreable_exposure),
        ("development", calendar_exposure, development_scoreable_exposure),
    )
    for portfolio in portfolios
])
if exposure_audit["scoreable_entity_days"].gt(
    exposure_audit["calendar_entity_days"] + 1e-9
).any():
    raise ValueError("Scoreable exposure exceeds calendar exposure")
display(exposure_audit)
affected_entities_per_fault = (
    intervals.assign(fault_id=intervals["fault_id"].astype(str))
    .groupby("fault_id")["entity_id"].nunique()
)
multi_entity_fault_ids = set(
    affected_entities_per_fault.loc[affected_entities_per_fault.gt(1)].index
)

workload_started = time.perf_counter()
for portfolio, channels in portfolios.items():
    admissible = []
    for quantile in quantiles:
        quantile = float(quantile)
        support = thresholds.loc[
            thresholds["model_id"].isin(channels),
            ["model_id", "threshold_quantile", "expected_tail_blocks"],
        ]
        support = support.loc[support["threshold_quantile"].eq(quantile)]
        support_ok = (
            len(support) == len(channels)
            and support["expected_tail_blocks"].ge(minimum_tail_blocks).all()
        )
        _, cases, _ = cases_from_grid(
            verification_grid, channels, quantile
        )
        scoreable_days = verification_scoreable_exposure[portfolio]
        rate = len(cases) / scoreable_days if scoreable_days else float("nan")
        _, upper = poisson_rate_interval(
            len(cases), scoreable_days, confidence
        )
        verification_workload[(portfolio, quantile)] = {
            "incidents": len(cases),
            "rate": rate,
            "ci_high": upper,
            "scoreable_exposure": scoreable_days,
            "calendar_exposure": verification_calendar_exposure,
            "score_availability": (
                scoreable_days / verification_calendar_exposure
                if verification_calendar_exposure else float("nan")
            ),
            "threshold_support_ok": support_ok,
            "minimum_expected_tail_blocks": (
                float(support["expected_tail_blocks"].min())
                if len(support) else 0.0
            ),
        }
        verification_rows.append({
            "portfolio": portfolio,
            "threshold_quantile": quantile,
            **verification_workload[(portfolio, quantile)],
        })
        if support_ok and np.isfinite(upper) and upper <= budget_gate:
            admissible.append(quantile)
    # Ascending quantiles run from most to least sensitive. The first is a
    # strictly label-free fallback; development may choose among all that
    # independently passed workload and threshold-support checks.
    admissible_quantiles[portfolio] = admissible
    label_free_quantile[portfolio] = min(admissible) if admissible else None
    print(
        f"workload {portfolio}: q={label_free_quantile[portfolio]} "
        f"({(time.perf_counter() - workload_started) / 60:.1f} min elapsed)"
    )

verification_workload_table = pd.DataFrame(verification_rows)
display(verification_workload_table)

development_quantiles = {
    portfolio: (
        admissible_quantiles[portfolio]
        if selection_eligible[portfolio]
        else ([label_free_quantile[portfolio]] if label_free_quantile[portfolio] is not None else [])
    )
    for portfolio in portfolios
}
needed_thresholds = {
    (channel, float(quantile))
    for portfolio, channels in portfolios.items()
    for quantile in development_quantiles[portfolio]
    for channel in channels
}
development_thresholds = thresholds.loc[[
    (str(row.model_id), float(row.threshold_quantile)) in needed_thresholds
    for row in thresholds.itertuples(index=False)
]].copy()
if development_thresholds.empty:
    raise RuntimeError(
        "No portfolio passed the label-free workload check; development truth "
        "must not be used to rescue an operating point."
    )
print(
    f"Building {len(development_thresholds)} required development "
    "channel/threshold alert sets"
)
alert_grid = alert_grid_from_score_file(
    score_path,
    development_thresholds,
    persistence=persistence,
    recovery_consecutive=RECOVERY_OBSERVATIONS,
    cadence_seconds=CADENCE_SECONDS,
    recovery_threshold_fraction=RECOVERY_THRESHOLD_FRACTION,
)

rows = []
candidate_results = {}
development_started = time.perf_counter()
for portfolio, channels in portfolios.items():
    for quantile in quantiles:
        quantile = float(quantile)
        if quantile not in development_quantiles[portfolio]:
            continue
        alerts, cases, members = cases_from_grid(
            alert_grid, channels, quantile
        )
        active_result = evaluate_cases(
            cases,
            members,
            events,
            intervals,
            exposure_value=development_scoreable_exposure[portfolio],
            exposure_unit="entity_day",
            decision_horizon_seconds=None,
            topology_memberships=topology,
            confidence_level=confidence,
        )
        prompt_result = evaluate_cases(
            cases,
            members,
            events,
            intervals,
            exposure_value=development_scoreable_exposure[portfolio],
            exposure_unit="entity_day",
            decision_horizon_seconds=POLICY["evaluation"]["default_decision_horizon_seconds"],
            topology_memberships=topology,
            confidence_level=confidence,
        )
        recall = metric_row(active_result, "event_recall")
        preimpact = metric_row(active_result, "preimpact_event_recall")
        prompt = metric_row(prompt_result, "event_recall")
        precision = metric_row(active_result, "case_precision")
        false_rate = metric_row(active_result, "false_cases_per_entity_day")
        multi_entity = active_result["fault_results"].loc[
            active_result["fault_results"]["fault_id"].astype(str).isin(
                multi_entity_fault_ids
            )
        ]
        localised = int(
            (multi_entity["detected"] & multi_entity["equivalent_scope"]).sum()
        )
        localisation_low, localisation_high = wilson_interval(
            localised, len(multi_entity), confidence
        )
        durations = (
            pd.to_datetime(alerts["alert_end"], utc=True)
            - pd.to_datetime(alerts["alert_start"], utc=True)
        ).dt.total_seconds() / 3600 if len(alerts) else pd.Series(dtype=float)
        workload = verification_workload[(portfolio, quantile)]
        candidate_key = f"{portfolio}|q={quantile:.6g}"
        candidate_results[candidate_key] = {
            "active": active_result, "prompt": prompt_result,
        }
        rows.append({
            "candidate_key": candidate_key,
            "portfolio": portfolio,
            "candidate": portfolio,
            "channels": ", ".join(channels),
            "threshold_quantile": quantile,
            "selection_eligible": selection_eligible[portfolio],
            "calibration_admissible": quantile in admissible_quantiles[portfolio],
            "label_free_choice": label_free_quantile[portfolio] == quantile,
            "verification_incidents": workload["incidents"],
            "verification_calendar_entity_days": workload["calendar_exposure"],
            "verification_scoreable_entity_days": workload["scoreable_exposure"],
            "verification_score_availability": workload["score_availability"],
            "verification_incidents_per_entity_day": workload["rate"],
            "verification_rate_ci_high": workload["ci_high"],
            "threshold_support_ok": workload["threshold_support_ok"],
            "minimum_expected_tail_blocks": workload["minimum_expected_tail_blocks"],
            "raw_alerts": len(alerts),
            "median_alert_hours": durations.median() if len(durations) else 0.0,
            "p95_alert_hours": durations.quantile(0.95) if len(durations) else 0.0,
            "incidents": len(cases),
            "development_calendar_entity_days": calendar_exposure,
            "development_scoreable_entity_days": development_scoreable_exposure[portfolio],
            "development_score_availability": (
                development_scoreable_exposure[portfolio] / calendar_exposure
                if calendar_exposure else float("nan")
            ),
            "scoreable_faults": int(recall["denominator"]),
            "event_recall": float(recall["value"]),
            "event_recall_ci_low": float(recall["ci_low"]),
            "event_recall_ci_high": float(recall["ci_high"]),
            "preimpact_scoreable_faults": int(preimpact["denominator"]),
            "preimpact_event_recall": float(preimpact["value"]),
            "preimpact_event_recall_ci_low": float(preimpact["ci_low"]),
            "preimpact_event_recall_ci_high": float(preimpact["ci_high"]),
            "prompt_scoreable_faults": int(prompt["denominator"]),
            "prompt_event_recall": float(prompt["value"]),
            "prompt_event_recall_ci_low": float(prompt["ci_low"]),
            "prompt_event_recall_ci_high": float(prompt["ci_high"]),
            "incident_precision": float(precision["value"]),
            "multi_entity_faults": len(multi_entity),
            "multi_entity_detected_and_localised": localised,
            "multi_entity_joint_detection_and_localisation_recall": (
                localised / len(multi_entity) if len(multi_entity) else float("nan")
            ),
            "multi_entity_joint_detection_and_localisation_recall_ci_low": localisation_low,
            "multi_entity_joint_detection_and_localisation_recall_ci_high": localisation_high,
            "false_incidents_per_entity_day": float(false_rate["value"]),
            "false_rate_ci_high": float(false_rate["ci_high"]),
            "false_incidents_per_entity_day_ci_high": float(false_rate["ci_high"]),
            "median_detection_delay_seconds": float(
                metric_row(active_result, "median_detection_delay_seconds")["value"]
            ),
            "maximum_missing_score_fraction": max(
                score_coverage[channel]["entity_p95"] for channel in channels
            ),
            "missing_score_fraction": max(
                score_coverage[channel]["entity_p95"] for channel in channels
            ),
        })
        print(
            f"development {portfolio} q={quantile:g}: "
            f"active LCB={float(recall['ci_low']):.3f}, "
            f"48h LCB={float(prompt['ci_low']):.3f} "
            f"({(time.perf_counter() - development_started) / 60:.1f} min elapsed)"
        )

comparison = pd.DataFrame(rows)
display(comparison.sort_values(
    ["false_incidents_per_entity_day", "event_recall"],
    ascending=[True, False],
))
print("Calibration-admissible development operating points:")
display(comparison.loc[comparison["calibration_admissible"], [
    "portfolio", "selection_eligible", "threshold_quantile",
    "threshold_support_ok", "minimum_expected_tail_blocks",
    "verification_calendar_entity_days",
    "verification_scoreable_entity_days", "verification_score_availability",
    "verification_incidents_per_entity_day", "verification_rate_ci_high",
    "event_recall", "event_recall_ci_low",
    "preimpact_event_recall", "preimpact_event_recall_ci_low",
    "prompt_event_recall", "prompt_event_recall_ci_low",
    "multi_entity_faults",
    "multi_entity_joint_detection_and_localisation_recall_ci_low",
    "false_incidents_per_entity_day_ci_high",
]])

## 5. Apply the fail-closed gate


In [ ]:
gate = POLICY["selection"]
preference = list(gate["eligible_portfolios"])
admissible_comparison = comparison.loc[
    comparison["calibration_admissible"] & comparison["selection_eligible"]
].copy()
# The existing 20% recall requirement is now applied to the conservative
# 95% Wilson lower bound. This is a methodology revision, not a refactor.
scoreable_denominators = comparison["scoreable_faults"].dropna().unique()
if len(scoreable_denominators) != 1:
    raise ValueError(
        "Candidates must share one scoreable-fault denominator; found "
        f"{scoreable_denominators.tolist()}"
    )
development_faults = int(scoreable_denominators[0])
def detections_needed(total, lower_bound):
    for detected in range(total + 1):
        if wilson_interval(detected, total, confidence)[0] >= lower_bound:
            return detected
    return None

recall_gate = float(gate["minimum_development_event_recall"])
gate_power = pd.DataFrame([
    {
        "scoreable_faults": total,
        "minimum_detected_faults": detections_needed(total, recall_gate),
        "minimum_observed_recall": (
            detections_needed(total, recall_gate) / total
        ),
        "required_wilson_lower_bound": recall_gate,
    }
    for total in sorted({int(gate["minimum_development_faults"]), development_faults})
])
display(gate_power)
selected, decision = select_development_candidate(
    admissible_comparison,
    development_faults=development_faults,
    false_incident_budget=POLICY["workload"]["false_incidents_per_entity_day"],
    budget_safety_factor=POLICY["workload"]["safety_factor"],
    minimum_faults=gate["minimum_development_faults"],
    minimum_recall_ci_low=gate["minimum_development_event_recall"],
    maximum_missing_score_fraction=gate["maximum_missing_score_fraction"],
    portfolio_preference=preference,
)

diagnostic_pool = admissible_comparison
if diagnostic_pool.empty:
    diagnostic_pool = comparison.loc[comparison["selection_eligible"]]
diagnostic = diagnostic_pool.sort_values(
    ["event_recall", "false_incidents_per_entity_day_ci_high"],
    ascending=[False, True],
).iloc[0]
def configuration(row, status, selection_basis):
    channels = row["channels"].split(", ")
    quantile = float(row["threshold_quantile"])

    def number(name):
        value = row[name]
        if pd.isna(value):
            return None
        integer_fields = {
            "scoreable_faults", "preimpact_scoreable_faults",
            "prompt_scoreable_faults", "verification_incidents",
        }
        return int(value) if name in integer_fields else float(value)

    localisation = (
        qualify_localisation(
            row,
            minimum_multi_entity_faults=gate["minimum_multi_entity_faults_for_localisation_claim"],
            minimum_joint_recall_ci_low=gate.get(
                "minimum_joint_localisation_recall_ci_low"
            ),
        )
        if selection_basis != "independent_late_calibration_workload"
        else {"status": "not_assessed_without_development_truth"}
    )
    early_warning = (
        qualify_early_warning(
            row,
            minimum_faults=gate["minimum_development_faults"],
            minimum_preimpact_recall_ci_low=gate["minimum_development_event_recall"],
            minimum_prompt_recall_ci_low=gate["minimum_development_event_recall"],
        )
        if selection_basis != "independent_late_calibration_workload"
        else {"status": "not_assessed_without_development_truth"}
    )
    return {
        "status": status,
        "selection_basis": selection_basis,
        "detection_window": "active_fault_interval",
        "prompt_detection_horizon_seconds": POLICY["evaluation"]["default_decision_horizon_seconds"],
        "candidate": str(row["candidate"]),
        "channels": channels,
        "threshold_quantile": quantile,
        "threshold_support": {
            "minimum_required_tail_blocks": minimum_tail_blocks,
            "minimum_observed_tail_blocks": number(
                "minimum_expected_tail_blocks"
            ),
            "passed": bool(row["threshold_support_ok"]),
        },
        "thresholds": {
            channel: float(thresholds.loc[
                thresholds["model_id"].eq(channel)
                & thresholds["threshold_quantile"].eq(quantile), "threshold"
            ].iloc[0])
            for channel in channels
        },
        "persistence_observations": {
            channel: int(persistence[channel]) for channel in channels
        },
        "recovery_observations": int(RECOVERY_OBSERVATIONS),
        "recovery_threshold_fraction": RECOVERY_THRESHOLD_FRACTION,
        "cadence_seconds": CADENCE_SECONDS,
        "incident_quiet_period_seconds": POLICY["incidents"]["quiet_period_seconds"],
        "verification_workload": {
            name: number(name)
            for name in (
                "verification_incidents",
                "verification_calendar_entity_days",
                "verification_scoreable_entity_days",
                "verification_score_availability",
                "verification_incidents_per_entity_day",
                "verification_rate_ci_high",
            )
        },
        "development_metrics": {
            name: number(name)
            for name in (
                "scoreable_faults", "event_recall", "event_recall_ci_low",
                "event_recall_ci_high", "preimpact_scoreable_faults",
                "preimpact_event_recall", "preimpact_event_recall_ci_low",
                "preimpact_event_recall_ci_high", "prompt_scoreable_faults",
                "prompt_event_recall", "prompt_event_recall_ci_low",
                "prompt_event_recall_ci_high", "incident_precision",
                "development_calendar_entity_days",
                "development_scoreable_entity_days",
                "development_score_availability",
                "false_incidents_per_entity_day", "false_rate_ci_high",
                "median_detection_delay_seconds", "maximum_missing_score_fraction",
            )
        },
        "early_warning_qualification": early_warning,
        "localisation_qualification": localisation,
        "model_run_id": MODEL_RUN_ID,
        "model_manifest_sha256": file_sha256(MODEL_ROOT / "model_manifest.json"),
        "resolved_policy_sha256": model_manifest["resolved_policy_sha256"],
        "selection_run_id": SELECTION_RUN_ID,
        "development_truth_manifest_sha256": file_sha256(
            TRUTH_ROOT / "truth_manifest.json"
        ),
        "evaluation_module_sha256": EVALUATION_MODULE_SHA256,
        "selection_module_sha256": SELECTION_MODULE_SHA256,
        "holdout_ready": early_warning["status"] == "qualified",
        "holdout_opened": False,
    }


label_free_rows = comparison.loc[comparison["label_free_choice"]].copy()
label_free_configurations = [
    configuration(
        row,
        "LABEL_FREE_CALIBRATION",
        "independent_late_calibration_workload",
    )
    for _, row in label_free_rows.iterrows()
]

early_warning = (
    qualify_early_warning(
        selected,
        minimum_faults=gate["minimum_development_faults"],
        minimum_preimpact_recall_ci_low=gate["minimum_development_event_recall"],
        minimum_prompt_recall_ci_low=gate["minimum_development_event_recall"],
    )
    if selected is not None
    else {"status": "not_assessed_no_detection_selection"}
)
holdout_ready = selected is not None and early_warning["status"] == "qualified"

if selected is not None:
    selected_results = candidate_results[str(selected["candidate_key"])]
    selected_fault_evidence = selected_results["active"]["fault_results"].copy()
    prompt_faults = selected_results["prompt"]["fault_results"][[
        "fault_id", "detected"
    ]].rename(columns={"detected": "prompt_detected"})
    selected_fault_evidence = selected_fault_evidence.merge(
        prompt_faults, on="fault_id", how="left", validate="one_to_one"
    )
    selected_fault_evidence["impact_known"] = selected_fault_evidence["impact_ts"].notna()
    selected_fault_summary = (
        selected_fault_evidence.groupby("fault_type", as_index=False)
        .agg(
            scoreable_faults=("fault_id", "nunique"),
            active_detected=("detected", "sum"),
            impact_known=("impact_known", "sum"),
            preimpact_detected=("preimpact", "sum"),
            prompt_detected=("prompt_detected", "sum"),
        )
    )
    selected_fault_summary["active_recall"] = (
        selected_fault_summary["active_detected"] / selected_fault_summary["scoreable_faults"]
    )
    selected_fault_summary["preimpact_recall"] = (
        selected_fault_summary["preimpact_detected"] / selected_fault_summary["impact_known"]
    )
    selected_fault_summary["prompt_recall"] = (
        selected_fault_summary["prompt_detected"] / selected_fault_summary["scoreable_faults"]
    )
    display(selected_fault_summary)
else:
    selected_fault_evidence = pd.DataFrame()
    selected_fault_summary = pd.DataFrame()

selection_status = {
    "model_manifest_sha256": file_sha256(MODEL_ROOT / "model_manifest.json"),
    "resolved_policy_sha256": model_manifest["resolved_policy_sha256"],
    "development_truth_manifest_sha256": file_sha256(
        TRUTH_ROOT / "truth_manifest.json"
    ),
    "evaluation_module_sha256": EVALUATION_MODULE_SHA256,
    "selection_module_sha256": SELECTION_MODULE_SHA256,
    **decision,
    "result": (
        "PASS_EARLY_WARNING" if holdout_ready
        else "PASS_DETECTION_ONLY" if selected is not None
        else "STOP"
    ),
    "reason": (
        ("Detection and early-warning evidence passed on development."
         if holdout_ready else
         "Detection passed on development; early-warning evidence remains unqualified.")
        if selected is not None
        else "No development candidate met every frozen gate; holdout remains sealed."
    ),
    "holdout_opened": False,
    "holdout_ready": holdout_ready,
    "detection_qualification": (
        "qualified_on_development" if selected is not None else "not_qualified"
    ),
    "early_warning_qualification": early_warning["status"],
    "early_warning_definition": (
        "after first observable evidence and before impact; "
        "prompt recall additionally requires detection within the registered horizon"
    ),
    "label_free_configurations": len(label_free_configurations),
    "recall_gate_statistic": "95% Wilson lower confidence bound",
    "localisation_qualification": (
        qualify_localisation(
            selected,
            minimum_multi_entity_faults=gate["minimum_multi_entity_faults_for_localisation_claim"],
            minimum_joint_recall_ci_low=gate.get(
                "minimum_joint_localisation_recall_ci_low"
            ),
        )["status"]
        if selected is not None else "not_assessed_no_detection_selection"
    ),
    "minimum_expected_tail_blocks": minimum_tail_blocks,
    "workload_verification_independent_of_threshold_fit": True,
}
display(pd.Series(selection_status, name="result").to_frame())

## 6. Publish only compact development evidence


In [ ]:
if OUTPUT_ROOT.exists():
    previous = read_json(OUTPUT_ROOT / "selection_status.json")
    require_same(
        previous,
        model_manifest_sha256=selection_status["model_manifest_sha256"],
        resolved_policy_sha256=selection_status["resolved_policy_sha256"],
        development_truth_manifest_sha256=(
            selection_status["development_truth_manifest_sha256"]
        ),
        evaluation_module_sha256=selection_status["evaluation_module_sha256"],
        selection_module_sha256=selection_status["selection_module_sha256"],
    )
    print("Using existing immutable selection:", previous["result"])
else:
    with immutable_output_directory(OUTPUT_ROOT) as output:
        comparison.to_parquet(output / "development_comparison.parquet", index=False)
        exposure_audit.to_parquet(output / "exposure_audit.parquet", index=False)
        verification_workload_table.to_parquet(
            output / "calibration_workload_audit.parquet", index=False
        )
        write_json(output / "selection_status.json", selection_status)
        write_json(
            output / "label_free_configurations.json",
            label_free_configurations,
        )
        write_json(output / "best_diagnostic_configuration.json", configuration(
            diagnostic, "DIAGNOSTIC_ONLY_NOT_DEPLOYABLE", "calibration_admissible_development_diagnosis"
        ))
        if selected is not None:
            write_json(output / "selected_configuration.json", configuration(
                selected,
                ("DEVELOPMENT_DETECTION_AND_EARLY_WARNING_GATES_PASSED"
                 if holdout_ready else "DEVELOPMENT_DETECTION_GATES_PASSED_EARLY_WARNING_PENDING"),
                "calibration_admissible_then_development_selection",
            ))
            selected_fault_evidence.to_parquet(
                output / "selected_fault_evidence.parquet", index=False
            )
            selected_fault_summary.to_parquet(
                output / "selected_fault_type_summary.parquet", index=False
            )

assert not selection_status["holdout_opened"]
if selected is None:
    print("STOP — no selected detection configuration was written")
    print("Use the diagnostic result for diagnosis only; do not open holdout truth")
else:
    print("PASS — one frozen detection configuration is ready for incident formation")
    print("Early warning:", selection_status["early_warning_qualification"])
    print("Localisation:", selection_status["localisation_qualification"])
    if holdout_ready:
        print("Holdout is eligible to be opened after the configuration is frozen")
    else:
        print("Holdout remains sealed: early-warning evidence is not yet qualified")
    print("Next: 08_ALERTS_INCIDENTS_AND_DYING_GASP.ipynb")
